### CONEXION DDBB OLIST

In [1]:
%pip install PyMySQL
from sqlalchemy import create_engine, text
import ssl

## CONEXION BBDD MYSQL ##
DB_USER = "nuclio"
DB_PASS = "nuclioTFM6"
DB_HOST = "nuclio.mysql.database.azure.com"
DB_NAME = "olist"

# Crear engine apuntando a la base 'olist'
engine = create_engine(
    f"mysql+pymysql://{DB_USER}:{DB_PASS}@{DB_HOST}:3306/{DB_NAME}?charset=utf8mb4",
    pool_pre_ping=True,
    connect_args={"ssl": {"cert_reqs": ssl.CERT_NONE, "check_hostname": False}} 
)

# tablas 'olist'
with engine.connect() as conn:
    tables = conn.execute(text("SHOW TABLES")).fetchall()
    tables = [row[0] for row in tables]   # convertir a lista simple de strings
    
    print("Tablas en la base 'olist':")
    for t in tables:
        print("-", t)



Note: you may need to restart the kernel to use updated packages.
Tablas en la base 'olist':
- dash_olist_categorias_resumen
- dash_olist_demorados
- dash_olist_sellers
- dash_olist_states
- dash_olist_ventas_meses
- dash_sentiment_analysis
- distribucion_pedidos
- olist_customers_dataset
- olist_geolocation_dataset
- olist_order_items_dataset
- olist_order_payments_dataset
- olist_order_reviews_dataset
- olist_orders_dataset
- olist_products_dataset
- olist_sellers_dataset
- pedidos_por_tiempo
- product_category_name_translation


In [2]:
import pandas as pd
from IPython.display import display, Markdown

# --- Cargar datos base desde la BBDD ---
query = """
SELECT 
    pct.product_category_name_english AS product_category_name,
    i.price,
    i.seller_id,
    c.customer_unique_id,
    o.order_id,
    o.order_purchase_timestamp,
    r.review_score
FROM olist_order_items_dataset i
LEFT JOIN olist_products_dataset p
    ON i.product_id = p.product_id
LEFT JOIN product_category_name_translation pct
    ON p.product_category_name = pct.product_category_name
LEFT JOIN olist_orders_dataset o
    ON i.order_id = o.order_id
LEFT JOIN olist_customers_dataset c
    ON o.customer_id = c.customer_id
LEFT JOIN olist_order_reviews_dataset r
    ON o.order_id = r.order_id
WHERE o.order_status <> 'canceled'
"""
df = pd.read_sql_query(query, con=engine)

# --- Procesamiento temporal ---
df["order_purchase_timestamp"] = pd.to_datetime(df["order_purchase_timestamp"], errors="coerce")
df["order_year"] = df["order_purchase_timestamp"].dt.year
df = df[df["order_year"].isin([2017, 2018])]

display(Markdown("Datos base cargados correctamente"))
display(df.head())


Datos base cargados correctamente

,product_category_name,price,seller_id,customer_unique_id,order_id,order_purchase_timestamp,review_score,order_year
0,bed_bath_table,29.99,1835b56ce799e6a4dc4eddc053f04066,570eb70ff97166b85ea96be3bfb65fef,d280ffa94ba8d94d4fc416ee85612a52,2017-11-09 14:07:54,NaN,2017
1,furniture_decor,79.90,0241d4d5d36f10f80c644447315af0bd,334fed5abcee3aa96c13f1432703e1fd,4ed7a5d31f58c9c3b20a61e3927db6d9,2018-05-07 23:25:09,NaN,2018
2,sports_leisure,42.00,7008613ea464bad5cb9b83456e1e6a8f,e46d6bcd22d95a9d167d807372d08add,5fc2475da462df87150a6013f0879f5b,2017-05-07 12:10:01,NaN,2017
3,sports_leisure,42.00,7008613ea464bad5cb9b83456e1e6a8f,e46d6bcd22d95a9d167d807372d08add,5fc2475da462df87150a6013f0879f5b,2017-05-07 12:10:01,NaN,2017
4,fashion_bags_accessories,59.00,6560211a19b47992c3666cc44a7e94c0,949b75abf6fdda78201547bf5f4c07f5,d8721b9f395286c3b43ff47a250ad40a,2018-08-02 22:09:44,NaN,2018


In [3]:
# --- DataFrame granular por vendedor, categoría y año ---
df_vendedores = (
    df.groupby(["product_category_name", "order_year", "seller_id"], dropna=False)
      .agg(
          TotalSales=("price", "sum"),
          OrdersQty=("order_id", "nunique"),
          Customers=("customer_unique_id", "nunique"),
          avg_score=("review_score", "mean")
      )
      .reset_index()
)

# --- Ticket medio por vendedor ---
df_vendedores["AvgTicket"] = (df_vendedores["TotalSales"] / df_vendedores["OrdersQty"]).round(2)
df_vendedores["avg_score"] = df_vendedores["avg_score"].round(2)

# --- Mostrar muestra del resultado ---
display(Markdown("#### Métricas por vendedor, categoría y año (2017–2018)"))
display(df_vendedores.head(10))

# --- Validación: número de vendedores únicos ---
total_sellers = df_vendedores["seller_id"].nunique()
display(Markdown(f"**Total de vendedores únicos en df_vendedores:** {total_sellers:,}"))


#### Métricas por vendedor, categoría y año (2017–2018)

,product_category_name,order_year,seller_id,TotalSales,OrdersQty,Customers,avg_score,AvgTicket
0,agro_industry_and_commerce,2017,06579cb253ecd5a3a12a9e6eb6bf8f47,89.90,1,1,NaN,89.90
1,agro_industry_and_commerce,2017,0ed6ce5d87fd9c69eaacaeb778d67235,39.90,1,1,NaN,39.90
2,agro_industry_and_commerce,2017,2528744c5ef5d955adc318720a94d2e7,4598.00,2,2,NaN,2299.00
3,agro_industry_and_commerce,2017,31ae0774c17fabd06ff707cc5bde005f,4029.92,5,5,NaN,805.98
4,agro_industry_and_commerce,2017,4e922959ae960d389249c378d1c939f5,132.00,3,3,NaN,44.00
5,agro_industry_and_commerce,2017,6481e96574816ead57975da2c0f6d80d,241.80,1,1,NaN,241.80
6,agro_industry_and_commerce,2017,6b2612338467c08c9b25f0cc55b1578d,92.90,1,1,NaN,92.90
7,agro_industry_and_commerce,2017,6bd69102ab48df500790a8cecfc285c2,5560.00,3,3,NaN,1853.33
8,agro_industry_and_commerce,2017,c9e5ad1f6647f5dbd1172adf6ae3cada,44.30,1,1,NaN,44.30
9,agro_industry_and_commerce,2017,d9e8c084b68fe958861d8f2c21202e6b,35.00,1,1,NaN,35.00


**Total de vendedores únicos en df_vendedores:** 3,029

In [4]:
# --- Total de vendedores únicos por año ---
unique_sellers_by_year = (
    df_vendedores.groupby("order_year")["seller_id"]
        .nunique()
        .reset_index(name="UniqueSellers")
        .sort_values("order_year")
)

# --- Total de vendedores únicos en todos los años ---
total_unique_sellers_all_years = df_vendedores["seller_id"].nunique()

# --- Mostrar resultados ---
display(Markdown("### Total de vendedores únicos por año (basado en df_vendedores)"))
display(unique_sellers_by_year)

display(Markdown(f"### Total de vendedores únicos en todos los años: **{total_unique_sellers_all_years:,}**"))


### Total de vendedores únicos por año (basado en df_vendedores)

,order_year,UniqueSellers
0,2017,1766
1,2018,2354


### Total de vendedores únicos en todos los años: **3,029**

In [6]:
from sqlalchemy import text
from sqlalchemy.types import Float, Integer, String

# --- Guardar DataFrame en la base de datos ---
table_name = "dash_olist_sellers"

# --- Eliminar tabla si existe (para evitar conflictos por reflexión) ---
with engine.begin() as conn:
    conn.execute(text(f"DROP TABLE IF EXISTS `{table_name}`;"))

# --- Definir tipos de columnas ---
dtype_map = {
    "product_category_name": String(100),
    "order_year": Integer(),
    "seller_id": String(50),
    "TotalSales": Float(),
    "OrdersQty": Integer(),
    "Customers": Integer(),
    "avg_score": Float(),
    "AvgTicket": Float(),
}

# --- Crear y cargar tabla ---
df_vendedores.to_sql(
    name=table_name,
    con=engine,
    if_exists="fail",     # evita sobreescritura accidental
    index=False,
    dtype=dtype_map,
    method="multi"
)

print(f"Tabla '{table_name}' creada y cargada correctamente en la base de datos.")

# --- Validación opcional: mostrar número de registros cargados ---
with engine.connect() as conn:
    result = conn.execute(text(f"SELECT COUNT(*) FROM `{table_name}`;"))
    total_rows = result.scalar()
    print(f"Total de registros insertados: {total_rows:,}")


Tabla 'dash_olist_sellers' creada y cargada correctamente en la base de datos.
Total de registros insertados: 8,333
